# data_quality-nyc_taxi-spark-iceberg

## 1. Overview

## 2. Setup

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote("sc://spark-connect:15002").getOrCreate()

## 3. Read

In [ ]:
df = spark.table("lakehouse.bronze.nyc_taxi_trips")

## 4. Transform

In [ ]:
rule = "fare_amount > 0 AND passenger_count BETWEEN 1 AND 6"
valid = df.where(rule)
quarantine = df.where(f"NOT ({rule}) OR fare_amount IS NULL")

## 5. Write

In [ ]:
valid.writeTo("lakehouse.silver.nyc_taxi_clean").using("iceberg").createOrReplace()
quarantine.writeTo("lakehouse.silver.nyc_taxi_quarantine").using("iceberg").createOrReplace()

## 6. Verify

In [ ]:
spark.sql("SELECT (SELECT count(*) FROM lakehouse.silver.nyc_taxi_clean) AS clean, (SELECT count(*) FROM lakehouse.silver.nyc_taxi_quarantine) AS quarantined").show(truncate=False)